# **Favorita 판매 데이터 탐색**

다음 주 수요를 계획하려면 판매가 **언제, 어디서, 얼마나 달라지는지** 먼저 알아야 합니다. 이 노트북은 매장·상품군의 판매 규모, 0 판매, 요일 반복, 판촉과 공휴일의 단순 차이를 확인합니다.

관찰된 패턴은 기준선과 예측 변수의 근거가 됩니다. 이곳의 집계는 설명용이며 판촉이나 휴일의 인과효과를 뜻하지 않습니다.

## **환경 준비**

Colab에서는 Drive를 연결합니다. 로컬과 Colab 모두 저장소 루트에서 실행합니다. 저장소 갱신은 별도의 00 노트북에서 마친 뒤 진행합니다.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

REPO = Path.cwd()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

## **패키지 준비**

Seaborn과 나눔고딕 글꼴을 설치합니다.


In [ ]:
%pip install -q seaborn koreanize-matplotlib


## **Library**


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

## **Utils**


In [ ]:
# 필요한 원본 열만 읽어 탐색용 표를 만듭니다.
def load_data(raw_path):
    train = pd.read_csv(
        raw_path / "train.csv",
        usecols=["date", "store_nbr", "family", "sales", "onpromotion"],
        parse_dates=["date"],
        dtype={"store_nbr": "int16", "family": "category", "sales": "float32", "onpromotion": "int16"},
    )
    stores = pd.read_csv(raw_path / "stores.csv")
    transactions = pd.read_csv(raw_path / "transactions.csv", parse_dates=["date"])
    return train, stores, transactions

In [ ]:
# 데이터 규모와 결측을 한 표로 요약합니다.
def data_summary(frame):
    return pd.DataFrame({
        "value": [
            len(frame),
            frame["date"].min().date(),
            frame["date"].max().date(),
            frame["store_nbr"].nunique(),
            frame["family"].nunique(),
            frame["sales"].isna().sum(),
            frame["onpromotion"].isna().sum(),
        ]
    }, index=["rows", "start_date", "end_date", "stores", "families", "missing_sales", "missing_promotion"])

In [ ]:
# 매장과 상품군을 합친 일별 판매량을 계산합니다.
def daily_sales(frame):
    return frame.groupby("date", as_index=False, observed=True)["sales"].sum()

In [ ]:
# 판촉 상품 유무에 따른 판매 분포를 기술적으로 비교합니다.
def promotion_summary(frame):
    return (
        frame.assign(promoted=frame["onpromotion"].gt(0))
        .groupby("promoted", observed=True)["sales"]
        .agg(rows="size", mean="mean", median="median")
    )

## **실행 설정**


In [ ]:
sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = REPO
RAW_PATH = PROJECT_ROOT / "data" / "raw"

In [ ]:
train, stores, transactions = load_data(RAW_PATH)
data_overview = data_summary(train)
data_overview


## **데이터 열을 읽는 법**

예시값은 원본 파일의 실제 행에서 가져왔습니다. 숫자로 저장된 매장 번호와 점포 군집은 크기보다 **구분**이 중요합니다.

### **핵심 판매 데이터: `train.csv`**

| 열 | 의미 | 예시값 | 예측에서의 역할·주의사항 |
|---|---|---|---|
| `date` | 판매일 | `2013-01-01` | 시간순 학습·검증에 사용 |
| `store_nbr` | 매장 번호 | `1` | 매장을 구분하며 숫자의 크기에는 의미가 없음 |
| `family` | 상품군 | `AUTOMOTIVE` | 매장×상품군 시계열을 구분 |
| `sales` | 해당 일의 판매량 | `0.0` | 목표일 판매량은 정답이며, 기준일 이전 값만 입력 가능 |
| `onpromotion` | 판촉 중인 상품 수 | `0` | 미래 계획을 미리 알 수 있을 때만 목표일 입력으로 사용 가능 |

### **매장·거래·휴일 데이터**

| 파일·열 | 의미 | 예시값 | 예측에서의 역할·주의사항 |
|---|---|---|---|
| `stores.store_nbr` | 매장 번호 | `1` | 판매 기록과 결합하는 키 |
| `stores.city` | 도시 | `Quito` | 지역 수요와 휴일을 연결 |
| `stores.state` | 주 | `Pichincha` | 지역 차이를 구분 |
| `stores.type` | 매장 유형 | `D` | 매장 특성의 대리 변수 |
| `stores.cluster` | 기존 유사 매장 번호 | `13` | 수치 크기의 순서로 해석하지 않음 |
| `transactions.transactions` | 하루 거래 건수 | `2111` | 해당 매장의 과거 수요 수준; 목표일 건수는 사용 불가 |
| `holidays_events.type` | 휴일·행사 유형 | `Holiday` | 달력 효과를 나타내는 변수 |
| `holidays_events.locale` | 적용 범위 | `Local` | 전국·지역·도시를 구분 |
| `holidays_events.locale_name` | 적용 지역 | `Manta` | 매장 지역과 연결 |
| `holidays_events.description` | 행사 설명 | `Fundacion de Manta` | 일부 이벤트만 파생 변수로 사용 |
| `holidays_events.transferred` | 휴일 이관 여부 | `False` | 이관된 원래 날짜를 실제 휴일로 세지 않음 |

## **판매량 분포**

판매량은 오른쪽 꼬리가 길기 때문에 원자료와 `log1p` 변환 결과를 함께 확인합니다.

In [ ]:
sample = train["sales"].sample(min(300_000, len(train)), random_state=42).clip(lower=0)
zero_rate = train["sales"].eq(0).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(sample.clip(upper=sample.quantile(0.99)), bins=60, ax=axes[0])
sns.histplot(np.log1p(sample), bins=60, ax=axes[1])
axes[0].set(title=f"판매량 분포 · 0 판매 {zero_rate:.1%}", xlabel="판매량")
axes[1].set(title="로그 변환 후 판매량", xlabel="log(1 + 판매량)")
fig.tight_layout()
plt.show()

## **전체 판매 추이와 요일 패턴**

In [ ]:
daily = daily_sales(train)
daily["rolling_28d"] = daily["sales"].rolling(28, min_periods=1).mean()
daily["dayofweek"] = daily["date"].dt.day_name().str[:3]
weekday_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.lineplot(data=daily, x="date", y="sales", alpha=0.25, linewidth=0.7, ax=axes[0])
sns.lineplot(data=daily, x="date", y="rolling_28d", linewidth=1.5, ax=axes[0])
sns.boxplot(data=daily, x="dayofweek", y="sales", order=weekday_order,
            showfliers=False, ax=axes[1])
axes[0].set(title="일별 판매량과 28일 평균", xlabel="", ylabel="판매량")
axes[1].set(title="요일별 판매량", xlabel="", ylabel="판매량")
fig.tight_layout()
plt.show()

## **주간 주기성 확인**

1일과 7일 간격 및 7일의 배수에서 자기상관을 비교합니다. 장기 추세로 상관이 과대평가되는 것을 줄이기 위해 28일 중심 이동평균을 제거한 결과도 함께 확인합니다. 중심 이동평균은 설명용 분석에만 사용하며 모델 입력에는 포함하지 않습니다.

In [ ]:
seasonal_lags = [1, 7, 14, 21, 28]
daily["trend_28d"] = daily["sales"].rolling(28, center=True, min_periods=14).mean()
daily["detrended_sales"] = daily["sales"] - daily["trend_28d"]
lag_correlation = pd.DataFrame({
    "lag": seasonal_lags,
    "raw_sales": [daily["sales"].autocorr(lag) for lag in seasonal_lags],
    "detrended_sales": [daily["detrended_sales"].autocorr(lag) for lag in seasonal_lags],
})
lag_overview = lag_correlation.set_index("lag").round(3)
lag_overview


In [ ]:
lag_plot = lag_correlation.melt(
    id_vars="lag", value_vars=["raw_sales", "detrended_sales"],
    var_name="계산 방식", value_name="상관계수",
)
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=lag_plot, x="lag", y="상관계수", hue="계산 방식", ax=ax)
ax.set(title="과거 며칠 전 판매와 오늘 판매의 상관", xlabel="간격 (일)")
fig.tight_layout()
plt.show()

전체 판매량에서 7일 자기상관은 0.826으로 1일의 0.767보다 높았습니다. 추세를 제거한 뒤에도 7일은 0.531, 14일은 0.608, 21일은 0.548, 28일은 0.602로 나타나 7일 단위의 반복 패턴이 유지됐습니다. 이를 모델에는 `sales_lag_7/14/28`, 7·14·28일 이동평균, 직전 주 동일 요일 판매량과 최근 4주 동일 요일 평균으로 반영합니다. 자기상관은 변수 설계의 근거이며 특정 요인의 인과효과를 의미하지는 않습니다.

## **상품군과 점포별 판매 규모**

In [ ]:
family_sales = (
    train.groupby("family", observed=True)["sales"].sum()
    .sort_values(ascending=False).head(15).sort_values()
)
store_sales = (
    train.groupby("store_nbr", observed=True)["sales"].sum()
    .sort_values(ascending=False).head(15).sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(x=family_sales.values, y=family_sales.index, ax=axes[0], color="steelblue")
sns.barplot(x=store_sales.values, y=store_sales.index.astype(str), ax=axes[1], color="steelblue")
axes[0].set(title="판매량 상위 상품군", xlabel="총판매량", ylabel="")
axes[1].set(title="판매량 상위 매장", xlabel="총판매량", ylabel="매장")
fig.tight_layout()
plt.show()

## **프로모션과 판매량**

아래 차이는 인과효과가 아니라 단순 집단 비교입니다. 상품군과 시점이 다르므로 모델 변수의 필요성을 판단하는 탐색 결과로만 사용합니다.

In [ ]:
promotion_overview = promotion_summary(train)
promotion_overview


In [ ]:
promotion_by_family = (
    train.assign(promoted=train["onpromotion"].gt(0))
    .groupby(["family", "promoted"], observed=True)["sales"].mean()
    .unstack()
    .dropna()
)
promotion_by_family["mean_difference"] = promotion_by_family[True] - promotion_by_family[False]
promotion_family_view = promotion_by_family.sort_values("mean_difference", ascending=False).head(15)
promotion_family_view


## **상품군별 0 판매 비율**

0 판매 비율이 높은 상품군은 절대오차보다 상대오차가 커질 수 있으므로 별도로 확인합니다.

In [ ]:
zero_rate_by_family = (
    train.assign(zero_sales=train["sales"].eq(0))
    .groupby("family", observed=True)["zero_sales"]
    .mean().sort_values(ascending=False)
)
top_zero = zero_rate_by_family.head(15).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=top_zero.values, y=top_zero.index, ax=ax, color="steelblue")
ax.set(title="0 판매 비율이 높은 상품군", xlabel="0 판매 비율", ylabel="")
fig.tight_layout()
plt.show()

## **국가 공휴일과 전체 판매량**

전국 단위 공휴일의 단순 비교이며 공휴일의 인과효과로 해석하지 않습니다.

In [10]:
holidays = pd.read_csv(RAW_PATH / "holidays_events.csv", parse_dates=["date"])
national_holiday_dates = holidays.loc[
    holidays["locale"].eq("National")
    & ~holidays["transferred"]
    & ~holidays["type"].eq("Work Day"),
    "date",
].unique()
holiday_sales = daily.assign(
    national_holiday=daily["date"].isin(national_holiday_dates)
).groupby("national_holiday")["sales"].agg(days="size", mean="mean", median="median")
holiday_sales

,days,mean,median
national_holiday,,,
False,1553,628031.6875,624711.3125
True,131,750471.6875,744956.9375


## **1차 확인 사항**

- 0 판매와 판매 규모 차이를 함께 보기 위해 RMSLE·MAE·WAPE를 나누어 해석합니다. 발주 목적에서는 7일 합계 WAPE와 Bias도 확인합니다.
- 요일과 장기 추세가 보여 달력 변수와 과거 판매량 변수가 필요합니다.
- 프로모션 집단의 차이는 크지만 인과관계로 해석하지 않습니다.
- 상품군과 점포 간 판매 규모가 달라 범주형 변수를 유지합니다.
- 0 판매가 많은 상품군과 국가 공휴일은 오류 분석에서 별도 구간으로 확인합니다.

## **EDA 핵심 결과 정리**

아래 숫자는 앞에서 계산한 전체 이력의 기술 통계를 한눈에 모은 것입니다. 0 판매 비율은 희소 수요를 따로 볼 필요가 있음을, 추세를 제거한 7일 자기상관은 동일 요일 기준선을 시험할 근거를 보여줍니다. 상위 상품군의 판매 비중은 전체 지표가 큰 상품군에 좌우될 수 있음을 설명합니다.

**전체 기간 EDA는 설명 자료**입니다. 군집 소속과 모델별 Test 성능을 연결할 때는 [02 수요 패턴 군집](02_demand_segments.ipynb)처럼 Valid 시작 전까지만 사용합니다. 판촉일과 판매량의 관계도 인과효과로 해석하지 않습니다.

In [ ]:
top_family = family_sales.index[-1]
top_family_share = family_sales.iloc[-1] / daily["sales"].sum()
weekly_correlation = lag_correlation.loc[
    lag_correlation["lag"].eq(7), "detrended_sales"
].iloc[0]

eda_summary = pd.DataFrame([
    {"관찰": "0 판매 비율", "값": f"{zero_rate:.1%}", "연결": "희소 수요의 오류를 별도 분석"},
    {"관찰": "추세 제거 후 7일 자기상관", "값": f"{weekly_correlation:.3f}",
     "연결": "동일 요일 기준선과 요일성 Feature"},
    {"관찰": f"최대 판매 상품군 ({top_family}) 비중", "값": f"{top_family_share:.1%}",
     "연결": "전체 WAPE와 상품군별 WAPE를 함께 보고"},
])
eda_summary